# GLM-5 and GLM-5.3-Flash

The third frontier architecture in this section, after [[kimi-k3]] and
[[deepseek-v4]] — and the most useful one to read third, because it shares
mechanisms with both and picks different trade-offs.

**Two sources, and they are not equal.** The [GLM-5 technical
report](https://arxiv.org/abs/2602.15763) is a real paper with ablations, and
everything in sections 1–5 below comes from reading it. **GLM-5.3-Flash**, the
model you may have arrived here from, has only a product documentation page — no
paper — so its specifics are flagged as such wherever they appear. That
distinction is kept visible throughout rather than blended away.

| | GLM-4.5 | GLM-5 | GLM-5.3-Flash |
|---|---|---|---|
| total / active params | 355B / 32B | **744B / 40B** | 320B / 18B *(docs)* |
| layers | — | 80 | 45 *(docs)* |
| experts | — | 256 | — |
| training tokens | — | 28.5T | 30T multimodal *(docs)* |
| attention | — | MLA → DSA | hybrid sparse + linear *(docs)* |

What makes GLM-5 worth studying is that the report **shows its failures**. It
publishes an ablation of five efficient-attention schemes, one of which is a
catastrophe, and that table explains the design better than the design does.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

banner("THE ABLATION THAT EXPLAINS THE ARCHITECTURE")

# GLM-5 technical report, Table 5. All variants continually trained on 190B
# tokens at 64K context from the same GLM-9B full-attention baseline, with a
# 1:1 ratio of efficient to full attention layers.
bench = ['RULER', 'MRCR', 'HELMET-ICL', 'RepoQA']
base   = {'RULER': (85.35, 75.28), 'MRCR': (36.53, 35.39),
          'HELMET-ICL': (77.68, 77.36), 'RepoQA': (69.00, 65.83)}
variants = {
    'SWA Interleave': {'RULER': (65.94, 44.93), 'MRCR': (30.03, 28.83),
                       'HELMET-ICL': (75.96, 63.52), 'RepoQA': (50.33, 39.33)},
    'SWA Pattern':    {'RULER': (83.72, 69.59), 'MRCR': (35.02, 33.58),
                       'HELMET-ICL': (76.48, 74.60), 'RepoQA': (62.33, 51.17)},
    'GDN':            {'RULER': (76.76, 64.00), 'MRCR': (31.72, 30.22),
                       'HELMET-ICL': (76.88, 74.84), 'RepoQA': (65.50, 56.17)},
    'SimpleGDN':      {'RULER': (81.76, 67.03), 'MRCR': (33.03, 31.27),
                       'HELMET-ICL': (79.80, 81.84), 'RepoQA': (65.50, 58.50)},
}

step("Every variant, measured against full attention at 128K")
print(f"  {'variant':>16}  " + "".join(f"{b:>13}" for b in bench)
      + f"  {'mean drop':>10}")
print("  " + "-" * 82)
print(f"  {'full attention':>16}  "
      + "".join(f"{base[b][1]:>13.2f}" for b in bench) + f"  {'--':>10}")
drops = {}
for name, v in variants.items():
    d = [v[b][1] - base[b][1] for b in bench]
    drops[name] = float(np.mean(d))
    print(f"  {name:>16}  " + "".join(f"{v[b][1]:>13.2f}" for b in bench)
          + f"  {np.mean(d):>+10.2f}")
print("  " + "-" * 82)

step("Ranked by average damage")
for name in sorted(drops, key=drops.get):
    bar = "#" * int(round(-drops[name] * 1.6))
    print(f"  {name:>16}  {drops[name]:>+7.2f}  {bar}")
print()
print(f"  The spread is the finding. Same baseline, same 190B tokens of")
print(f"  adaptation, same 1:1 layer ratio -- and the worst scheme loses")
print(f"  {-drops['SWA Interleave']:.1f} points on average while the best loses"
      f" {-drops[min(drops, key=drops.get)]:.1f}.")

step("Why fixed sliding windows fail, and search fixes them")
print(f"  Both SWA rows use the SAME mechanism and the SAME budget:")
print(f"  half the layers windowed at 4096 tokens, half full attention.")
print(f"  The only difference is WHICH layers get windowed.")
print()
print(f"  {'context':>9}  {'full attn':>11}  {'interleaved':>12}  {'searched':>10}")
print("  " + "-" * 48)
ruler = {'4K': (95.19, 94.87, 95.78), '8K': (93.67, 54.02, 92.54),
         '16K': (92.01, 25.89, 88.92), '32K': (91.09, 12.61, 82.52),
         '64K': (85.35, 8.32, 70.23), '128K': (75.28, 6.51, 53.95)}
for k, (f, i, s) in ruler.items():
    print(f"  {k:>9}  {f:>11.2f}  {i:>12.2f}  {s:>10.2f}")
print("  " + "-" * 48)
print(f"  Interleaved SWA collapses to {ruler['128K'][1]:.2f} at 128K -- worse than")
print(f"  guessing on a retrieval benchmark. The searched pattern, same layer")
print(f"  budget, holds {ruler['128K'][2]:.2f}.")
print()
print(f"  The pattern the beam search found:")
print(f"    SFSSFFSSSFFFFSSFSFFFFFFSFSFSSFSSFSFSSFSSS")
print(f"    (S = sliding window, F = full attention, 40 layers)")
pat = "SFSSFFSSSFFFFSSFSFFFFFFSFSFSSFSSFSFSSFSSS"
print(f"    S: {pat.count('S')} layers, F: {pat.count('F')} layers")
print(f"  Nothing about it is periodic. That is the point -- the assumption")
print(f"  behind interleaving is that layers are interchangeable, and they")
print(f"  are not. Note also the search ran only at 16K yet generalised to")
print(f"  128K.")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
  THE ABLATION THAT EXPLAINS THE ARCHITECTURE

Every variant, measured against full attention at 128K
           variant          RULER         MRCR   HELMET-ICL       RepoQA   mean drop
  ----------------------------------------------------------------------------------
    full attention          75.28        35.39        77.36        65.83          --
    SWA Interleave          44.93        28.83        63.52        39.33      -19.31
       SWA Pattern          69.59        33.58        74.60        51.17       -6.23
               GDN          64.00        30.22        74.84        56.17       -7.16
         SimpleGDN          67.03        31.27        81.84        58.50       -3.80
  ----------------------------------------------------------------------------------

Ranked by average damage
    SWA Interleave   -19.31  ###############################
               GDN    -7.16  ######

## 1. Why GLM-5 chose DSA over all of them

The paper's conclusion from that table is worth quoting in substance: every
scheme above **discards information** and pays for it on fine-grained retrieval,
even with half the layers kept dense. Sliding windows throw away everything
outside the window. Linear attention compresses history into a fixed-size state,
so old detail is gone whether you needed it or not.

**DSA is different in kind.** Its lightning indexer scores every past token
cheaply and keeps the top-$k$. Nothing is structurally excluded — a token from a
million positions back is retrievable if the indexer ranks it highly. The paper
calls this *lossless by construction*, and it is why DSA can be applied to **all**
layers while the others need a 1:1 mix with full attention.

That mechanism is worked through in detail in [[deepseek-v4]] §3 — GLM-5 adopts
it rather than inventing it. What GLM-5 contributes is **how to get there
cheaply**.

In [2]:
banner("RETROFITTING SPARSE ATTENTION INSTEAD OF PRE-TRAINING IT")

step("The two-stage recipe, and its cost")
print("  Stage 1, warm-up: train ONLY the indexer, base model frozen")
print(f"    1,000 steps x 14 sequences x 202,752 tokens, lr 5e-3")
warm = 1000 * 14 * 202_752
print(f"    = {warm/1e9:.2f}B tokens")
print("  Stage 2, sparse adaptation: co-train model and indexer")
print(f"    = 20B tokens")
total_glm = warm/1e9 + 20
print(f"  GLM-5 total     : {total_glm:.1f}B tokens")
print(f"  DeepSeek-V3.2   : 943.7B tokens")
print(f"  ratio           : {943.7/total_glm:.0f}x cheaper")
print()
print(f"  Same mechanism, same reported end quality. The saving is entirely")
print(f"  in not re-learning what the dense model already knew.")

step("The GLM-4.7-Flash check: how much does the warm-up alone buy?")
print("  A separate small-scale test. Two stages: (i) train ONLY the indexer")
print("  for 1,000 steps at batch size 16 with the base model frozen, then")
print("  (ii) co-train model and indexer on 150B tokens.")
print()
print("  Only the figures the paper states exactly are shown here -- the")
print("  128K pair, and the deltas after full joint training:")
print()
print(f"  {'stage':>34}  {'RULER @128K':>12}")
print("  " + "-" * 50)
print(f"  {'dense baseline':>34}  {79.21:>12.2f}")
print(f"  {'indexer warm-up only':>34}  {71.35:>12.2f}")
print("  " + "-" * 50)
print(f"  {'drop from freezing the base model':>34}  {71.35-79.21:>+12.2f}")
print()
print("  After the 150B-token joint phase, relative to the dense baseline:")
print(f"  {'context':>10}  {'delta':>8}")
print("  " + "-" * 22)
for ctx, d in [("16K", +0.86), ("32K", +0.49), ("64K", +1.72), ("128K", -0.35)]:
    print(f"  {ctx:>10}  {d:>+8.2f}")
print("  " + "-" * 22)
print(f"  mean delta over those four: "
      f"{np.mean([0.86, 0.49, 1.72, -0.35]):+.2f}")
print()
print("  So the indexer alone, with every base weight frozen, already holds")
print("  most of the quality, and the damage is concentrated at the longest")
print("  context. Joint training then closes it -- and IMPROVES on the dense")
print("  baseline at 16K, 32K and 64K, leaving only a 0.35 deficit at 128K.")
print()
print("  That is the argument for retrofitting: the model does not need to")
print("  relearn language, only how to use the indexer.")

  RETROFITTING SPARSE ATTENTION INSTEAD OF PRE-TRAINING IT

The two-stage recipe, and its cost
  Stage 1, warm-up: train ONLY the indexer, base model frozen
    1,000 steps x 14 sequences x 202,752 tokens, lr 5e-3
    = 2.84B tokens
  Stage 2, sparse adaptation: co-train model and indexer
    = 20B tokens
  GLM-5 total     : 22.8B tokens
  DeepSeek-V3.2   : 943.7B tokens
  ratio           : 41x cheaper

  Same mechanism, same reported end quality. The saving is entirely
  in not re-learning what the dense model already knew.

The GLM-4.7-Flash check: how much does the warm-up alone buy?
  A separate small-scale test. Two stages: (i) train ONLY the indexer
  for 1,000 steps at batch size 16 with the base model frozen, then
  (ii) co-train model and indexer on 150B tokens.

  Only the figures the paper states exactly are shown here -- the
  128K pair, and the deltas after full joint training:

                               stage   RULER @128K
  ------------------------------------------

## 2. Muon Split — orthogonalise each head separately

This is GLM-5's most interesting contribution and it connects directly to
[[deepseek-v4]] §7, where Muon's Newton–Schulz orthogonalisation was worked
through.

Recall what Muon does: replace the gradient $M = U\Sigma V^T$ with $UV^T$, so
every direction gets an equally-sized step. The question GLM-5 asks is **what
counts as one matrix**.

The standard recipe orthogonalises the whole up-projection $W^{UQ}$ at once. But
that matrix is a stack of per-head blocks, and orthogonalising the stack couples
them: the shared $\Sigma$ is dominated by whichever head had the largest
gradient, so a head with small gradients gets a step scaled by another head's
magnitude.

**Muon Split** slices the matrix by head and orthogonalises each block
independently, letting each head's weights update at its own scale. The effect
in the paper's ablation is not subtle — it is what makes MLA competitive with
GQA-8 at all:

| variant | MMLU | C-Eval | RACE | HumanEval |
|---|---|---|---|---|
| GQA-8 | 61.2 | 60.0 | 79.6 | 38.5 |
| MLA | 61.5 | 59.7 | **77.8** | **33.5** |
| MLA + Muon Split | **62.5** | **62.1** | **79.9** | 36.7 |
| MLA-256 + Muon Split | 62.0 | 59.9 | 79.6 | 36.6 |

There is a second consequence the paper notes: with Muon Split, attention logits
stay stable through pre-training **without any clipping**. Kimi needed QK-Clip;
[[deepseek-v4]] avoided it with QK RMSNorm; GLM-5 gets it from the optimiser.
Three papers, three routes around the same failure.

In [3]:
banner("MUON SPLIT: WHOLE-MATRIX vs PER-HEAD ORTHOGONALISATION")

rng = np.random.RandomState(0)
n_heads, d_head, d_model = 4, 8, 16

step("A gradient whose heads have very different magnitudes")
scales = np.array([3.0, 1.0, 0.3, 0.05])
blocks = [rng.randn(d_head, d_model) * s for s in scales]
W = np.vstack(blocks)                      # the stacked up-projection
print(f"  W is ({W.shape[0]} x {W.shape[1]}) = {n_heads} head blocks of"
      f" ({d_head} x {d_model})")
print(f"  {'head':>6}  {'||block||_F':>12}  {'relative':>10}")
print("  " + "-" * 32)
for h, b in enumerate(blocks):
    print(f"  {h:>6}  {np.linalg.norm(b):>12.4f}"
          f"  {np.linalg.norm(b)/np.linalg.norm(blocks[0]):>10.4f}")
print("  " + "-" * 32)
print(f"  head 3's gradient is {np.linalg.norm(blocks[0])/np.linalg.norm(blocks[3]):.0f}x"
      f" smaller than head 0's.")

def orth(M):
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    return U @ Vt

step("Standard Muon: orthogonalise the whole stack")
O_whole = orth(W)
print(f"  {'head':>6}  {'||update block||_F':>19}  {'share of total':>15}")
print("  " + "-" * 44)
tot = np.linalg.norm(O_whole)
for h in range(n_heads):
    blk = O_whole[h*d_head:(h+1)*d_head]
    print(f"  {h:>6}  {np.linalg.norm(blk):>19.4f}"
          f"  {np.linalg.norm(blk)**2/tot**2:>15.4f}")
print("  " + "-" * 44)
w_norms = [np.linalg.norm(O_whole[h*d_head:(h+1)*d_head]) for h in range(n_heads)]
print(f"  spread across heads: {max(w_norms)/min(w_norms):.2f}x")
print(f"  The head with the smallest gradient still receives the smallest")
print(f"  update -- the whole point of orthogonalisation was undone by")
print(f"  sharing one set of singular values across all four heads.")

step("Muon Split: orthogonalise each head block on its own")
O_split = np.vstack([orth(b) for b in blocks])
print(f"  {'head':>6}  {'||update block||_F':>19}  {'share of total':>15}")
print("  " + "-" * 44)
tot_s = np.linalg.norm(O_split)
for h in range(n_heads):
    blk = O_split[h*d_head:(h+1)*d_head]
    print(f"  {h:>6}  {np.linalg.norm(blk):>19.4f}"
          f"  {np.linalg.norm(blk)**2/tot_s**2:>15.4f}")
print("  " + "-" * 44)
s_norms = [np.linalg.norm(O_split[h*d_head:(h+1)*d_head]) for h in range(n_heads)]
print(f"  spread across heads: {max(s_norms)/min(s_norms):.4f}x")
print(f"  every block now has {d_head} singular values equal to 1, so each")
print(f"  contributes sqrt({d_head}) = {np.sqrt(d_head):.4f} to the norm regardless of")
print(f"  how large its gradient was.")

step("Check the singular values directly")
sv_whole = np.linalg.svd(O_whole[3*d_head:4*d_head], compute_uv=False)
sv_split = np.linalg.svd(O_split[3*d_head:4*d_head], compute_uv=False)
print(f"  head 3 (the weakest gradient), singular values of its update block:")
print(f"    whole-matrix Muon : {np.round(sv_whole, 4)}")
print(f"    Muon Split        : {np.round(sv_split, 4)}")
print(f"    all ones under Split: {np.allclose(sv_split, 1.0)}")
print()
print(f"  This is what 'projection weights for different attention heads")
print(f"  update at different scales' means, and why splitting fixes it.")

  MUON SPLIT: WHOLE-MATRIX vs PER-HEAD ORTHOGONALISATION

A gradient whose heads have very different magnitudes
  W is (32 x 16) = 4 head blocks of (8 x 16)
    head   ||block||_F    relative
  --------------------------------
       0       35.5606      1.0000
       1       10.4636      0.2942
       2        3.3350      0.0938
       3        0.5867      0.0165
  --------------------------------
  head 3's gradient is 61x smaller than head 0's.

Standard Muon: orthogonalise the whole stack
    head   ||update block||_F   share of total
  --------------------------------------------
       0               2.7964           0.4887
       1               2.3533           0.3461
       2               1.6020           0.1604
       3               0.2752           0.0047
  --------------------------------------------
  spread across heads: 10.16x
  The head with the smallest gradient still receives the smallest
  update -- the whole point of orthogonalisation was undone by
  sharing one 

## 3. MLA-256 — moving cost from decoding to training

MLA compresses keys and values into a latent, which is why its cache is small.
The cost lands somewhere else: during **decoding**, MLA's dot product runs in the
latent dimension — 576 for DeepSeek-V3 — against GQA's 128. Decoding is
memory-bandwidth-bound and token-by-token, so that is exactly the wrong place to
be expensive.

GLM-5's fix: **raise the head dimension from 192 to 256 and use fewer heads.**
Parameter count and training compute both scale with $n_h \times d_{\text{head}}$,
so holding that product fixed leaves training untouched. Decoding cost scales
with $n_h$ alone — so fewer, fatter heads is strictly cheaper to decode.

The paper says heads decrease "by 1/3". Worth checking what the product
constraint actually requires, because the two do not quite agree — the notebook
computes both readings rather than picking one.

In [4]:
banner("THE HEAD-SHAPE TRADE")

d_old, d_new, h_old = 192, 256, 128
print(f"  head dimension {d_old} -> {d_new}   (x{d_new/d_old:.4f})")
print()
print(f"  Parameters and training FLOPs scale with n_h x d_head.")
print(f"  Decoding cost scales with n_h (one latent dot product per head).")
print()
print(f"  {'reading':>34}  {'n_h':>6}  {'n_h x d':>9}  {'vs old':>8}  {'decode':>8}")
print("  " + "-" * 72)
print(f"  {'original':>34}  {h_old:>6}  {h_old*d_old:>9,}  {1.0:>8.4f}  {1.0:>8.4f}")
for lab, h_new in [("heads x 3/4 (product constant)", int(h_old*3/4)),
                   ("heads x 2/3 ('decrease by 1/3')", int(h_old*2/3))]:
    print(f"  {lab:>34}  {h_new:>6}  {h_new*d_new:>9,}"
          f"  {h_new*d_new/(h_old*d_old):>8.4f}  {h_new/h_old:>8.4f}")
print("  " + "-" * 72)
print(f"  Only the x3/4 reading holds parameters exactly constant, which is")
print(f"  what the paper claims. The x2/3 reading would shrink the model by")
print(f"  {1 - (int(h_old*2/3)*d_new)/(h_old*d_old):.1%}. Either way decoding gets cheaper by 25-33%,")
print(f"  which is the direction that matters.")

step("Where the decoding cost actually goes")
d_latent, d_gqa = 576, 128
print(f"  per generated token, per layer, dot-product work:")
for lab, h, d in [("MLA (DeepSeek-V3 shape)", h_old, d_latent),
                  ("MLA-256 (heads x 3/4)", int(h_old*3/4), d_latent),
                  ("GQA-8 for reference", 8, d_gqa)]:
    print(f"    {lab:>26}: {h:>3} heads x {d:>3} dims = {h*d:>7,}")
print()
print(f"  MLA's cache is small and its decode arithmetic is not. GLM-5 keeps")
print(f"  the cache saving and buys back {1-0.75:.0%} of the arithmetic.")
print()
print(f"  The paper's justification for why this was available at all: the")
print(f"  head count in DeepSeek-V3 was tuned to the roofline of one specific")
print(f"  accelerator (H800), and is not optimal on different hardware.")

  THE HEAD-SHAPE TRADE
  head dimension 192 -> 256   (x1.3333)

  Parameters and training FLOPs scale with n_h x d_head.
  Decoding cost scales with n_h (one latent dot product per head).

                             reading     n_h    n_h x d    vs old    decode
  ------------------------------------------------------------------------
                            original     128     24,576    1.0000    1.0000
      heads x 3/4 (product constant)      96     24,576    1.0000    0.7500
     heads x 2/3 ('decrease by 1/3')      85     21,760    0.8854    0.6641
  ------------------------------------------------------------------------
  Only the x3/4 reading holds parameters exactly constant, which is
  what the paper claims. The x2/3 reading would shrink the model by
  11.5%. Either way decoding gets cheaper by 25-33%,
  which is the direction that matters.

Where the decoding cost actually goes
  per generated token, per layer, dot-product work:
       MLA (DeepSeek-V3 shape): 128 he

## 4. Multi-token prediction with shared parameters

MTP predicts several tokens ahead. It improves the base model, and the extra
heads double as a **draft model for speculative decoding**: propose $n$ tokens
cheaply, verify them in one pass, keep the prefix that matches.

The tension: predicting $n$ tokens ahead normally needs $n$ MTP layers, so memory
grows linearly in speculative depth. DeepSeek-V3 trains **one** layer and
predicts two tokens at inference — cheap, but the second token is produced by a
layer never trained for that position, and acceptance suffers.

GLM-5 **shares one set of parameters across three MTP layers**. Memory matches
the single-layer cost; training matches the inference-time use.

The payoff is the accept length — how many drafted tokens survive verification on
average. GLM-5: **2.76** against DeepSeek-V3.2's **2.55**, at the same
speculative depth of 4. Small-looking, and worth converting into what it means.

In [5]:
banner("SPECULATIVE DECODING: WHAT ACCEPT LENGTH BUYS")

step("Accept length -> speedup")
print("  With accept length L, one verification pass emits L tokens.")
print("  Speedup over plain decoding is L, minus the drafting overhead.")
print()
print(f"  {'model':>18}  {'accept length':>14}  {'raw speedup':>12}"
      f"  {'at 20% overhead':>17}")
print("  " + "-" * 68)
for lab, L in [("DeepSeek-V3.2", 2.55), ("GLM-5", 2.76)]:
    print(f"  {lab:>18}  {L:>14.2f}  {L:>12.2f}x  {L/1.2:>16.2f}x")
print("  " + "-" * 68)
print(f"  relative gain: {2.76/2.55 - 1:+.1%} more tokens per verification pass")

step("Where accept length comes from")
print("  If each drafted token is accepted independently with probability p,")
print("  and drafting depth is D, the expected accepted prefix is")
print("      L = sum_{i=1..D} p^i  (plus 1 for the always-correct token)")
print()
D = 4
print(f"  {'p':>6}  {'expected prefix':>16}  {'+1 verified':>12}")
print("  " + "-" * 38)
for p in [0.5, 0.6, 0.7, 0.8, 0.9]:
    L = sum(p**i for i in range(1, D+1))
    print(f"  {p:>6.2f}  {L:>16.4f}  {L+1:>12.4f}")
print("  " + "-" * 38)

# what per-token acceptance rate reproduces each reported accept length?
from scipy.optimize import brentq
def solve_p(target, D=4):
    f = lambda p: 1 + sum(p**i for i in range(1, D+1)) - target
    return brentq(f, 1e-9, 0.999999)
p_ds, p_glm = solve_p(2.55), solve_p(2.76)
print(f"  Inverting for the reported numbers at depth {D}:")
print(f"    DeepSeek-V3.2 L={2.55:.2f}  ->  per-token acceptance p = {p_ds:.4f}")
print(f"    GLM-5         L={2.76:.2f}  ->  per-token acceptance p = {p_glm:.4f}")
print(f"    difference: {p_glm-p_ds:+.4f} per token")
print()
print(f"  Verify by substituting back:")
for lab, p, tgt in [("DeepSeek", p_ds, 2.55), ("GLM-5", p_glm, 2.76)]:
    L = 1 + sum(p**i for i in range(1, D+1))
    print(f"    {lab:>9}: 1 + sum p^i = {L:.4f}  target {tgt:.2f}"
          f"   MATCH: {np.isclose(L, tgt)}")
print()
print(f"  A {p_glm-p_ds:+.3f} improvement in per-token acceptance compounds over the")
print(f"  draft into {2.76/2.55-1:+.1%} throughput. That is the return on training the")
print(f"  MTP layers in the configuration they are actually used in.")

  SPECULATIVE DECODING: WHAT ACCEPT LENGTH BUYS

Accept length -> speedup
  With accept length L, one verification pass emits L tokens.
  Speedup over plain decoding is L, minus the drafting overhead.

               model   accept length   raw speedup    at 20% overhead
  --------------------------------------------------------------------
       DeepSeek-V3.2            2.55          2.55x              2.12x
               GLM-5            2.76          2.76x              2.30x
  --------------------------------------------------------------------
  relative gain: +8.2% more tokens per verification pass

Where accept length comes from
  If each drafted token is accepted independently with probability p,
  and drafting depth is D, the expected accepted prefix is
      L = sum_{i=1..D} p^i  (plus 1 for the always-correct token)

       p   expected prefix   +1 verified
  --------------------------------------
    0.50            0.9375        1.9375
    0.60            1.3056        2.

## 5. GLM-5.3-Flash — what the documentation claims

Everything above is from the technical report. **GLM-5.3-Flash has no paper**,
so this section is a reading of the product documentation and should be treated
accordingly.

The headline: 320B total / 18B active across 45 layers, 1M context, natively
multimodal over images, video and files, trained on a 30T-token multimodal
corpus. Two mechanisms are named:

- **IndexShare** — one indexer reused across every four sparse-attention layers, rather than one per layer. *(Not in the paper; from Z.ai material.)*
- **IndexPool** — four indexer key vectors compressed into one by weighted pooling.

Both attack the same overhead. DSA's indexer is cheap per token but it runs in
every layer over the whole context, and at a million tokens that stops being
negligible. The claimed effect is **3.01× less attention computation and 4.44×
smaller KV cache** relative to GLM-5.3.

It also uses **Manifold-Constrained Hyper-Connections** — the same mHC worked
through in [[deepseek-v4]] §2, Sinkhorn projection onto the Birkhoff polytope and
all. Two independent frontier labs adopting the same residual-stream mechanism
within months is worth noticing.

The arithmetic below reconstructs where savings of that size could come from. It
is a plausibility check on stated numbers, not a reproduction — the docs give no
configuration to reproduce.

In [6]:
banner("RECONSTRUCTING THE GLM-5.3-FLASH EFFICIENCY CLAIMS")

print("  Source: Z.ai documentation. No paper, no configuration published.")
print("  Treat the reconstruction as an order-of-magnitude check.")

L_FLASH, N_CTX = 45, 1_000_000

step("IndexShare: one indexer per four layers instead of one per layer")
print(f"  {L_FLASH} layers, indexer runs over the full {N_CTX:,}-token context")
for share in [1, 2, 4]:
    n_idx = int(np.ceil(L_FLASH / share))
    print(f"    reuse across {share} layer(s): {n_idx:>3} indexer passes"
          f"   ({L_FLASH/n_idx:.2f}x fewer)")
print()
print(f"  Sharing across 4 layers removes ~{1-1/4:.0%} of indexer work on its own.")

step("IndexPool: four key vectors pooled into one")
print(f"  The indexer scores every compressed entry. Pooling its KEYS 4:1")
print(f"  quarters both the entries it scores and what must be stored:")
print(f"    {'':>22}  {'entries at 1M':>15}  {'relative':>10}")
print("  " + "-" * 52)
print(f"    {'no pooling':>22}  {N_CTX:>15,}  {1.0:>10.2f}")
print(f"    {'IndexPool 4:1':>22}  {N_CTX//4:>15,}  {0.25:>10.2f}")
print("  " + "-" * 52)

step("Combined, against the stated figures")
share_gain, pool_gain = 4.0, 4.0
print(f"  If the two applied independently and to everything, the ceiling")
print(f"  would be {share_gain:.0f}x and {pool_gain:.0f}x respectively. The documentation reports:")
print()
print(f"    {'attention computation':>24}  3.01x")
print(f"    {'KV cache size':>24}  4.44x")
print()
print(f"  3.01x sits below the 4x indexer ceiling, which is what you would")
print(f"  expect: IndexShare only touches the indexer, and the core attention")
print(f"  over the selected top-k is unchanged.")
print(f"  4.44x EXCEEDS the 4x pooling ratio, so the cache saving cannot come")
print(f"  from IndexPool alone -- the hybrid linear-attention layers store a")
print(f"  fixed-size state rather than a growing cache, and that is the most")
print(f"  likely source of the remainder.")
print()
print(f"  Stated as an inference, not a result: the numbers are consistent")
print(f"  with the described mechanisms, and nothing published lets anyone")
print(f"  check them properly.")

step("Active-parameter trend across the family")
print(f"  {'model':>16}  {'total':>8}  {'active':>8}  {'active %':>9}")
print("  " + "-" * 48)
for lab, tot, act in [("GLM-4.5", 355, 32), ("GLM-5", 744, 40),
                      ("GLM-5.3-Flash", 320, 18)]:
    print(f"  {lab:>16}  {tot:>7}B  {act:>7}B  {act/tot:>9.1%}")
print("  " + "-" * 48)
print(f"  Sparsity is rising across the family: GLM-4.5 activated"
      f" {32/355:.1%},")
print(f"  GLM-5 {40/744:.1%}, Flash {18/320:.1%}. Compare [[deepseek-v4]]:")
print(f"  V4-Flash activates {13/284:.1%} and V4-Pro {49/1600:.1%}. The whole field is")
print(f"  moving the same direction -- more total capacity, less of it")
print(f"  touched per token.")

  RECONSTRUCTING THE GLM-5.3-FLASH EFFICIENCY CLAIMS
  Source: Z.ai documentation. No paper, no configuration published.
  Treat the reconstruction as an order-of-magnitude check.

IndexShare: one indexer per four layers instead of one per layer
  45 layers, indexer runs over the full 1,000,000-token context
    reuse across 1 layer(s):  45 indexer passes   (1.00x fewer)
    reuse across 2 layer(s):  23 indexer passes   (1.96x fewer)
    reuse across 4 layer(s):  12 indexer passes   (3.75x fewer)

  Sharing across 4 layers removes ~75% of indexer work on its own.

IndexPool: four key vectors pooled into one
  The indexer scores every compressed entry. Pooling its KEYS 4:1
  quarters both the entries it scores and what must be stored:
                              entries at 1M    relative
  ----------------------------------------------------
                no pooling        1,000,000        1.00
             IndexPool 4:1          250,000        0.25
  -------------------------------

## 6. Three architectures, side by side

Having now worked through [[kimi-k3]], [[deepseek-v4]] and GLM-5, the
comparison is the most useful thing on this page.

| | Kimi K3 | DeepSeek-V4 | GLM-5 |
|---|---|---|---|
| long-context attention | Kimi Delta Attention (linear, gated decay) | CSA + HCA (compress, then select) | DSA (select, no compression) |
| how it got there | trained in | trained in | **retrofitted** by continued pre-training |
| residual stream | attention residuals across blocks | **mHC** | standard |
| optimiser fix for logits | QK-Clip | QK RMSNorm | **Muon Split** |
| MoE | Stable LatentMoE | DeepSeekMoE, hash routing early | 256 experts |

Three labs, three answers to one problem, and the disagreements are informative:

**On compression.** DeepSeek compresses KV entries and then selects among them.
GLM-5's ablation argues compression is exactly what costs you retrieval accuracy,
and that selection alone — lossless by construction — is preferable. Both ship.

**On stability.** All three hit exploding attention logits. Kimi clips them,
DeepSeek normalises them away, GLM-5 removes the cause by changing what the
optimiser treats as one matrix. The third is the only one that costs nothing at
inference.

**On getting there.** GLM-5's cheapest idea is not an architecture at all. DSA
was retrofitted onto a dense model for about 22.8B tokens where DeepSeek-V3.2
spent 943.7B — over 40× cheaper, for the same mechanism at reportedly the same
quality.

### Verified in this notebook

- The efficient-attention ablation reproduced and ranked; interleaved SWA collapses to 6.51 on RULER@128K while the searched pattern, with an identical layer budget, holds 53.95
- Continued-pretraining cost computed from the report's own figures: ~22B tokens against DeepSeek-V3.2's 943.7B
- Muon Split contrasted with whole-matrix orthogonalisation: the weakest head's update block carries singular values far from 1 under standard Muon and exactly 1 under Split
- The MLA-256 head-shape trade computed under both readings of "decrease by 1/3", showing which one holds parameters constant
- Accept lengths 2.55 and 2.76 inverted to per-token acceptance probabilities and substituted back to confirm
- The GLM-5.3-Flash efficiency claims checked for consistency against the mechanisms described, with the part that does not follow from them identified

### A note on sources

Sections 1–4 and 6 come from the [GLM-5 technical
report](https://arxiv.org/abs/2602.15763), read directly. Section 5 comes from
Z.ai's GLM-5.3-Flash documentation, which is a product page — no architecture
paper exists for that model, so nothing there is independently checkable, and the
notebook says so rather than presenting it at the same confidence as the rest.